# TDKPS — Detecting Perspective Shifts in Multi-Agent Systems

**Goal**: After this session, you will be able to implement the TDKPS pipeline from memory: omnibus distance matrix → classical MDS → agent-level drift scores → a calibrated group-level permutation test.
**Time**: 30 minutes.

## What and Why

You operate a fleet of **black-box** agents — no weights, no logits, just responses. TDKPS (Temporal Data Kernel Perspective Space) monitors behavioral drift by querying every agent with a fixed probe set at each time snapshot, embedding the responses with a frozen text embedder, and **jointly** embedding all (agent, time) snapshots into one shared "perspective space". Movement in that space = behavior change; permutation tests turn "it moved" into a calibrated p-value. The paper uses this to detect the effect of a real exogenous event on 99 simulated digital congresspersons queried across 14 timepoints.

## Key Formula

Classical (Torgerson) MDS of the omnibus distance matrix $\tilde D \in \mathbb{R}^{TN \times TN}$:

$$B = -\tfrac{1}{2}\, J \,(\tilde D \circ \tilde D)\, J, \qquad J = I_{TN} - \tfrac{1}{TN}\mathbf{1}\mathbf{1}^\top, \qquad \psi = V_d\, \Lambda_d^{1/2}$$

**Where:**
- $\tilde D_{ij}$ — Frobenius distance between the replicate-averaged response matrices of snapshots $i$ and $j$; row index $i = t\cdot N + n$ ranges over **all** (timepoint, agent) pairs
- $\circ$ — elementwise product (so $\tilde D \circ \tilde D$ is the entrywise-squared distance matrix)
- $J$ — the centering matrix (subtracts row/column means; puts the centroid at the origin)
- $\Lambda_d, V_d$ — top-$d$ eigenvalues/eigenvectors of $B$
- $\psi_i \in \mathbb{R}^d$ — the TDKPS position of agent $n$ at time $t$

## References
[arXiv:2512.05013](https://arxiv.org/abs/2512.05013) — Bridgeford & Helm, *Detecting Perspective Shifts in Multi-agent Systems* (Sec. 2–3: method; Sec. 4: temporal Gaussian blobs simulation, which the setup cell below reproduces).

In [ ]:
import torch
import matplotlib.pyplot as plt
from jaxtyping import Float
from torch import Tensor

torch.manual_seed(0)

# --- Simulation: "temporal Gaussian blobs" (paper Sec. 4) ------------------------
# N black-box agents are probed with M fixed queries at T=2 snapshots, R replicate
# responses each; every response is embedded with a frozen embedder. We simulate the
# embeddings directly. Agents 0..19 (signal class) genuinely change behavior between
# t=0 and t=1; agents 20..39 (null class) are static.
T, N, M, R, p = 2, 40, 8, 20, 32   # timepoints, agents, queries, replicates, embed dim
p_s = 5                             # behavior lives in a small signal subspace
tau = 0.8                           # effect size of the temporal shift (paper's tau)
sigma_eps, sigma_agent = 0.3, 0.75  # response noise / stable per-agent "personality"

signal_idx = torch.arange(0, N // 2)   # agents whose behavior shifts
null_idx   = torch.arange(N // 2, N)   # agents whose behavior is static

# class mean: signal class drifts front-loaded -> back-loaded across the signal dims
front = torch.exp(-0.5 * torch.arange(p_s, dtype=torch.float32))
back  = front.flip(dims=[0])
mu = torch.zeros(T, 2, p)                       # (time, class, p); class 0 = signal
mu[0, 0, :p_s] = front
mu[1, 0, :p_s] = (1 - tau) * front + tau * back

# stable agent effect eta_n: the agent's persistent identity, constant across time
eta = torch.zeros(N, p)
eta[:, :p_s] = sigma_agent * torch.randn(N, p_s)

# per-query orthogonal map O_m: each query "views" the latent behavior differently
O = torch.linalg.qr(torch.randn(M, p, p)).Q     # (M, p, p), each slice orthogonal

# responses[t, n, m, r] = O_m @ (mu_class + eta_n) + noise
# same role as: embedder(agent n's r-th response to query q_m at snapshot t) in R^p
cls = (torch.arange(N) >= N // 2).long()        # 0 = signal class, 1 = null class
latent = mu[:, cls, :] + eta                    # (T, N, p)
responses = torch.einsum("mij,tnj->tnmi", O, latent)[:, :, :, None, :] \
          + sigma_eps * torch.randn(T, N, M, R, p)
print("responses:", tuple(responses.shape), "= (T, N, M, R, p)")

## Part 1: Omnibus Distance Matrix

Average the replicates, then compute the distance between **every pair of (timepoint, agent) snapshots** — all $T\cdot N$ of them in one matrix. Entry $(i,j)$ is the Frobenius distance $\|\bar X_i - \bar X_j\|_F$ between the two snapshots' replicate-averaged $(M, p)$ response matrices. Row/column index convention: $i = t\cdot N + n$ (all of $t{=}0$ first, then all of $t{=}1$).

**Insight**: embedding all $T\cdot N$ rows *jointly* (the "omnibus" trick) is what makes positions comparable across time — two separate per-timepoint embeddings would each come with their own arbitrary rotation, making "movement" meaningless.

**Predict before you code**: What shape is the output? Commit it as the literal on the first line of the next cell. Also: which block of the matrix (within-$t_0$, within-$t_1$, or cross-time) do you expect to have the largest entries for signal agents?

In [ ]:
predicted_D_shape = (..., ...)  # commit a shape before you code

def build_omnibus_distance(
    responses: Float[Tensor, "T N M R p"],
) -> Float[Tensor, "TN TN"]:
    """Omnibus distance matrix over all (timepoint, agent) snapshots.

    Row/col index i = t * N + n. Entry (i, j) = ||Xbar_i - Xbar_j||_F where
    Xbar is the snapshot's replicate-averaged (M, p) response matrix.
    """
    # Implement from memory
    pass

In [ ]:
# --- Part 1 Validation ---
D_omni = build_omnibus_distance(responses)

# Prediction reveal (no pass/fail -- the gap is the signal)
print(f"  You predicted shape {predicted_D_shape}; actual {tuple(D_omni.shape)}")

assert D_omni.shape == (T * N, T * N), f"Shape: expected {(T*N, T*N)}, got {tuple(D_omni.shape)}"
print(f"  Shape: {tuple(D_omni.shape)} -- correct")
assert torch.allclose(D_omni, D_omni.T, atol=1e-3), "a distance matrix must be symmetric"
assert D_omni.diagonal().abs().max() < 0.05, (
    "self-distance must be ~0 (D[i,i] = ||Xbar_i - Xbar_i||). Note: float32 cdist computes "
    "distances via matmul and leaves ~1e-3 noise on the diagonal -- that much is fine.")
print(f"  Symmetric, zero diagonal -- correct")
print(f"  Range: [{D_omni.min():.3f}, {D_omni.max():.3f}], mean off-diag: {D_omni.sum() / (D_omni.numel() - T*N):.3f}")

# Reference match: naive per-entry computation on random entries
import random as _random
_random.seed(0)
_Xbar = responses.mean(dim=3)
max_diff = 0.0
for _ in range(25):
    t1_, n1_ = _random.randrange(T), _random.randrange(N)
    t2_, n2_ = _random.randrange(T), _random.randrange(N)
    ref = torch.linalg.norm(_Xbar[t1_, n1_] - _Xbar[t2_, n2_])   # Frobenius norm
    got = D_omni[t1_ * N + n1_, t2_ * N + n2_]
    max_diff = max(max_diff, abs(float(ref - got)))
assert max_diff < 0.02, (
    f"Max diff vs naive per-entry Frobenius distance: {max_diff:.2e}. "
    "Check your row ordering (i = t*N + n) and that you averaged over the replicate dim. "
    "(float32 cdist noise alone is ~1e-3; a real bug is orders of magnitude larger.)")
print(f"  Reference match vs naive loop -- correct (max diff: {max_diff:.2e})")

# Diagnostic: block structure. Cross-time block, signal agents only, should be largest.
w0 = D_omni[:N, :N][signal_idx][:, signal_idx].mean()
w1 = D_omni[N:, N:][signal_idx][:, signal_idx].mean()
x01 = D_omni[:N, N:][signal_idx][:, signal_idx].mean()
print(f"  Signal-agent block means -- within t0: {w0:.3f}, within t1: {w1:.3f}, cross-time: {x01:.3f}")

print("\nPart 1 complete.")

## Part 2: Classical MDS (Torgerson)

Turn the omnibus distance matrix into coordinates: `classical_mds(D, d)` returns $\psi \in \mathbb{R}^{n\times d}$ whose pairwise Euclidean distances approximate $D$. Steps: double-center the squared distances into $B$ (Key Formula above), eigendecompose $B$, keep the top-$d$ eigenpairs, and scale eigenvectors by $\sqrt{\lambda}$.

**Insight**: double-centering works because $\|x_i - x_j\|^2 = \|x_i\|^2 + \|x_j\|^2 - 2\langle x_i, x_j\rangle$ — the centering matrix $J$ on both sides kills the norm terms and leaves exactly the Gram matrix $B = X_c X_c^\top$ of the *centered* configuration. *Self-explanation (one line, don't skip)*: why must the centroid be pinned at the origin before inner products are recoverable from distances alone?

**Predict before you code**: How many eigenvalues of $B$ will be "large" (> 1% of the biggest)? Think about where the simulation actually puts signal. Commit your count on the first line of the next cell. Also note: what order does `torch.linalg.eigh` return eigenvalues in?

In [ ]:
predicted_num_large_eigs = ...  # commit a count before you code

def classical_mds(
    D: Float[Tensor, "n n"], d: int
) -> Float[Tensor, "n d"]:
    """Torgerson classical MDS: coordinates in R^d from an n x n distance matrix."""
    # Implement from memory
    pass

In [ ]:
# --- Part 2 Validation ---
d_embed = 10   # generous; the paper picks d by an eigenvalue-elbow (profile likelihood)
psi = classical_mds(D_omni, d_embed)
psi_full = classical_mds(D_omni.double(), T * N)   # full-rank, for the exactness check

# Prediction reveal: column energies of the full embedding ARE the MDS eigenvalues
_eigs = psi_full.pow(2).sum(dim=0)
_n_large = int((_eigs > 0.01 * _eigs.max()).sum())
print(f"  You predicted {predicted_num_large_eigs} large eigenvalues; actual {_n_large}"
      f"  (top 8: {[round(float(e), 1) for e in _eigs[:8]]})")

assert psi.shape == (T * N, d_embed), f"Shape: expected {(T*N, d_embed)}, got {tuple(psi.shape)}"
print(f"  Shape: {tuple(psi.shape)} -- correct")
assert torch.isfinite(psi_full).all(), (
    "NaNs in the embedding: eigh returns tiny NEGATIVE eigenvalues for a rank-deficient "
    "PSD matrix (numerical noise), and sqrt(negative) = NaN. Clamp eigenvalues to min 0.")
assert psi.mean(dim=0).abs().max() < 1e-3, (
    "columns must have zero mean -- double-centering pins the centroid at the origin")
print(f"  Centered (max |col mean| = {psi.mean(dim=0).abs().max():.2e}) -- correct")

_col_e = psi.pow(2).sum(dim=0)
assert (_col_e[:-1] >= _col_e[1:] - 1e-4).all(), (
    "column energies must be DESCENDING -- you must take the TOP-d eigenpairs, "
    "but torch.linalg.eigh returns eigenvalues in ascending order")
print(f"  Top-d eigenpairs in descending order -- correct")

# Reference match: D is Euclidean by construction, so the full-rank embedding must
# reproduce it (near-)exactly.
recon = torch.cdist(psi_full, psi_full)
max_diff = (recon - D_omni.double()).abs().max()
assert max_diff < 0.05, (
    f"Full-rank CMDS must reconstruct a Euclidean D. Max |cdist(psi) - D| = {max_diff:.3f}. "
    "Check the -1/2 factor, that you squared D entrywise, and that J multiplies on BOTH sides.")
print(f"  Full-rank reconstruction: max |cdist(psi_full) - D| = {max_diff:.2e} -- correct")
print(f"  Variance captured by top {d_embed}: {float(_eigs[:d_embed].sum() / _eigs.sum()):.1%}")

print("\nPart 2 complete.")

## Part 3: Agent-Level Drift Scores

The paper's agent-level test statistic: $\delta_n = \|\psi_n^{(t_1)} - \psi_n^{(t_0)}\|_2$ — how far agent $n$ moved in perspective space. With the row convention $i = t\cdot N + n$ (and $T{=}2$), agent $n$'s two snapshots are rows $n$ and $N{+}n$.

**Predict before you code**: signal agents shifted their class mean; null agents only wiggle by residual noise. Roughly what will the ratio of mean drift (signal / null) be? Commit a number on the first line of the next cell.

In [ ]:
predicted_drift_ratio = ...  # commit a number before you code

def drift_scores(psi: Float[Tensor, "TN d"]) -> Float[Tensor, "N"]:
    """Per-agent drift delta_n = ||psi_n^(t1) - psi_n^(t0)||_2. Assumes T=2, rows t*N+n."""
    # Implement from memory
    pass

In [ ]:
# --- Part 3 Validation ---
delta = drift_scores(psi)

_ratio = float(delta[signal_idx].mean() / delta[null_idx].mean())
print(f"  You predicted a drift ratio of {predicted_drift_ratio}; actual {_ratio:.2f}"
      f"  (signal mean {delta[signal_idx].mean():.3f}, null mean {delta[null_idx].mean():.3f})")

assert delta.shape == (N,), f"Shape: expected {(N,)}, got {tuple(delta.shape)}"
assert (delta >= 0).all(), "a norm cannot be negative"
print(f"  Shape {tuple(delta.shape)}, nonnegative -- correct")
assert _ratio > 2.0, (
    f"Signal agents genuinely moved; ratio {_ratio:.2f} <= 2 suggests wrong row pairing "
    "(agent n's snapshots are rows n and N+n) or a distance taken along the wrong dim.")
_sep = float((delta[signal_idx] > delta[null_idx].max()).float().mean())
print(f"  Signal/null separation: ratio {_ratio:.2f}, "
      f"{_sep:.0%} of signal agents drift beyond the largest null drift")
assert _sep > 0.8, "with tau=0.8 nearly every signal agent should out-drift every null agent"

# The picture worth the whole pipeline: arrows t0 -> t1 in the first two MDS dims
fig, ax = plt.subplots(figsize=(6.5, 5))
for n in range(N):
    a, b = psi[n, :2], psi[N + n, :2]
    c = "#c1392b" if n < N // 2 else "#9a9a9a"
    ax.annotate("", xy=(float(b[0]), float(b[1])), xytext=(float(a[0]), float(a[1])),
                arrowprops=dict(arrowstyle="->", color=c, lw=1.1, alpha=0.85))
ax.scatter(psi[:N, 0], psi[:N, 1], s=12, c=["#c1392b"] * (N // 2) + ["#9a9a9a"] * (N // 2), zorder=3)
ax.set_xlabel("TDKPS dim 1"); ax.set_ylabel("TDKPS dim 2")
ax.set_title("Perspective drift, t0 → t1  (red = signal class, gray = null class)")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

print("\nPart 3 complete.")

## Part 4: Group-Level Test — Energy Distance + Paired Permutation

Did the *signal group as a whole* shift? Test statistic: the **energy distance** between the group's two point clouds $A = \{\psi_n^{(t_0)}\}$ and $B = \{\psi_n^{(t_1)}\}$,

$$\mathcal{E}(A, B) = 2\,\overline{\|a - b\|} \;-\; \overline{\|a - a'\|} \;-\; \overline{\|b - b'\|}$$

(plain means over all pairs, diagonal included — the V-statistic). Null distribution: under $H_0$ each agent's two snapshots are exchangeable, so **swap each agent's $t_0/t_1$ embeddings independently with probability ½**, recompute $\mathcal{E}$, repeat; $p = (1 + \#\{\mathcal{E}_{perm} \ge \mathcal{E}_{obs}\}) / (1 + n_{perm})$.

**Insight**: the obvious statistic — mean paired drift $\overline{\delta_n}$ — is *invariant* to swapping a pair ($\|a-b\| = \|b-a\|$), so its permutation distribution is a point mass and the test is vacuous ($p \approx 1$ always). Energy distance works because its **cross-agent** terms ($\|a_i - b_j\|$, $i \ne j$) do change under swaps. *Self-explanation (one line)*: why does exchangeability of $(\psi_n^{(t_0)}, \psi_n^{(t_1)})$ under $H_0$ justify exactly this swap and not, say, shuffling agents across the group?

**Predict before you code**: what is $\mathcal{E}(A, A)$ — a cloud against itself? Commit a number on the first line of the next cell.

In [ ]:
predicted_selfdist = ...  # commit a number before you code: energy_distance(A, A)

def energy_distance(
    A: Float[Tensor, "na d"], B: Float[Tensor, "nb d"]
) -> Tensor:
    """V-statistic energy distance between two point clouds (formula above)."""
    # Implement from memory
    pass

def paired_permutation_pvalue(
    psi_t0: Float[Tensor, "ng d"], psi_t1: Float[Tensor, "ng d"], n_perm: int = 500
) -> float:
    """P-value for H0: no group-level shift, via per-agent t0/t1 swaps (procedure above)."""
    # Implement from memory
    pass

In [ ]:
# --- Part 4 Validation ---
A_sig, B_sig = psi[signal_idx], psi[N + signal_idx]
A_nul, B_nul = psi[null_idx], psi[N + null_idx]

_selfd = float(energy_distance(A_sig, A_sig))
print(f"  You predicted E(A, A) = {predicted_selfdist}; actual {_selfd:.6f}")

assert abs(_selfd) < 1e-5, (
    "energy distance of a cloud with itself must be exactly 0: the cross term 2*E||a-b|| "
    "equals the two within terms it subtracts")
assert torch.isclose(energy_distance(A_sig, B_sig), energy_distance(B_sig, A_sig), atol=1e-5), \
    "energy distance must be symmetric in its arguments"
print(f"  E(A,A)=0 and symmetry -- correct")
print(f"  Observed statistic -- signal group: {float(energy_distance(A_sig, B_sig)):.4f}, "
      f"null group: {float(energy_distance(A_nul, B_nul)):.4f}")

torch.manual_seed(123)
p_signal = paired_permutation_pvalue(A_sig, B_sig)
torch.manual_seed(123)
p_null = paired_permutation_pvalue(A_nul, B_nul)
print(f"  p-value -- signal group: {p_signal:.4f}, null group: {p_null:.4f}")

assert p_signal < 0.05, (
    f"p_signal = {p_signal:.3f}: the signal group's shift is enormous -- the observed energy "
    "distance should beat essentially every swap-permuted value. Check the >= direction and "
    "that you swap BOTH clouds consistently per agent.")
assert p_null > 0.1, (
    f"p_null = {p_null:.3f}: the null group is static, so rejecting here means your null "
    "distribution is broken. Classic cause: a statistic invariant under the swap (p would "
    "pin to 1.0) or permuting only one of the two clouds.")
assert 0 < p_signal <= 1 and 0 < p_null <= 1, "the +1 correction keeps p in (0, 1]"
print(f"  Signal group rejected, null group retained -- correct")

print("\nPart 4 complete.")

## Session Debrief

Write your answers into the code cell below (typing is overt retrieval — far stronger
than answering "in your head"). Don't scroll up.

1. Write the double-centering formula of classical MDS, and say what matrix $B$ *is* (in terms of the point configuration).
2. Why can't the group-level permutation test use mean paired drift $\overline{\|\psi_n^{(t_1)} - \psi_n^{(t_0)}\|}$ as its statistic?
3. What does one row of the omnibus distance matrix index, and why embed all $T\cdot N$ rows jointly instead of one MDS per timepoint?

**Check yourself**: your worked solution is in `_solutions/tdkps.ipynb` — open it (and the
paper) to grade your answers. Opening it ends the retrieval rep, so answer first.

**Challenge**: Close this notebook, open a blank one, and rewrite Part 2 (`classical_mds`)
from scratch without looking back.

In [ ]:
debrief = """
1.
2.
3.
"""  # type your recall here before checking _solutions/

In [ ]:
# --- Session log: fill the `___` then run (appends one line to .practice-log.jsonl) ---
import json, datetime, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / ".git").exists() or (p / ".spaced-repetition.json").exists())
record = {
    "problem": "papers/tdkps-perspective-shifts",
    "date": datetime.date.today().isoformat(),
    "budget_min": 30,
    "actual_min": "___",                                # how long it really took (number)
    "parts": [                                          # outcome: unaided | hint | solution | failed
        {"n": 1, "tier": 3, "outcome": "___"},          # omnibus distance matrix
        {"n": 2, "tier": 3, "outcome": "___"},          # classical MDS
        {"n": 3, "tier": 3, "outcome": "___"},          # drift scores
        {"n": 4, "tier": 3, "outcome": "___"},          # energy distance + permutation
    ],
    "difficulty": "___",                                # 1 (trivial) .. 5 (over my head)
    "stuck": "___",                                     # one phrase: where you got stuck
}
assert "___" not in json.dumps(record), "fill every ___ before running this cell"
with open(_root / ".practice-log.jsonl", "a") as f:
    f.write(json.dumps(record) + "\n")
print("logged ->", _root / ".practice-log.jsonl")

## Extension (Optional)

Try one of these variations:
- **Power curve**: wrap the whole pipeline in a function of $\tau$, sweep $\tau \in \{0, 0.1, \dots, 0.5\}$ (regenerating data each time), and plot the signal-group p-value vs $\tau$. Where does detection die?
- **Break it on purpose**: replace the omnibus MDS with a *separate* `classical_mds` per timepoint and recompute drift scores. Explain why the arrows in the Part 3 plot become meaningless (what freedom does each independent embedding have?).
- **U- vs V-statistic**: change `energy_distance` to exclude the diagonals of the within-cloud terms (the U-statistic). Do the p-values move? Why is the permutation test's validity indifferent to the choice?

## Anki Cards

Add these to your deck:

**Card 1**
Front: You have an $n\times n$ pairwise distance matrix and need point coordinates (classical MDS). Formula for the Gram matrix?
Back: $B = -\frac{1}{2} J D^{\circ 2} J$

**Card 2**
Front: TDKPS: why one joint MDS over all $T\cdot N$ (agent, time) rows instead of a separate embedding per timepoint?
Back: shared axes — cross-time comparable

**Card 3**
Front: Your paired permutation test returns $p \approx 1$ no matter what. Statistic was mean paired distance — what's wrong?
Back: swap-invariant statistic